# ROGII - Wellbore Geology Prediction - Stage 4a submission notebook

Self-contained: no internet access needed (Code Competition requirement).

**Stage 4a: global gradient-boosted model.** Trains one `HistGradientBoostingRegressor`
across every training well's real evaluation zone (each well already carries a real,
labeled eval zone - `TVT_input` is `NaN` there but the true `TVT` is known), using the
Stage 2 linear prior, a Stage 3 windowed GR/typewell shape-match, geometry, and
eval-zone-position features as inputs. The model learns per-region how much to trust each
signal instead of a hand-tuned threshold.

Locally verified with 5-fold GroupKFold-by-well (leak-free): **overall RMSE 52.90**, vs.
67.09 for the linear prior alone and 71.98 for the windowed GR-match alone (both worse -
see `../context/05-plan-of-attack.md` Stage 3/4a for the full negative-result writeup that
led here). Still far from the ~4.86 public-LB leader - this is a flat/tabular model over
per-row features, not a true sequence model (Stage 4b, still ahead).

This notebook trains the FINAL model on all training data (no CV split - the CV above was
just for validation) and predicts the real test wells.

In [ ]:
import glob
import os
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

RANDOM_STATE = 42
HALF_WINDOW = 5
SEARCH_EXTRA_FT = 100.0
ACCEPT_SLACK = 1.5

# Kaggle mounts Code Competition data at /kaggle/input/competitions/<slug>/ (NOT
# /kaggle/input/<slug>/, which is where an attached Dataset lands instead - confirmed
# from a live Kaggle session's own os.walk('/kaggle/input') output on this competition).
_KAGGLE_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
DATA_DIR = _KAGGLE_DIR if os.path.isdir(_KAGGLE_DIR) else os.path.join("..", "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"DATA_DIR = {DATA_DIR}")
assert os.path.isdir(TRAIN_DIR), f"train dir not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"test dir not found: {TEST_DIR}"

In [ ]:
def list_wells(split_dir):
    files = glob.glob(os.path.join(split_dir, "*__horizontal_well.csv"))
    return sorted(os.path.basename(f).split("__")[0] for f in files)


def linear_prior_predict(known, eval_rows):
    if len(known) < 2:
        fallback = known["TVT_input"].mean() if len(known) else np.nan
        return np.full(len(eval_rows), fallback)
    model = LinearRegression()
    model.fit(known[["MD", "Z"]], known["TVT_input"])
    return model.predict(eval_rows[["MD", "Z"]])


def windowed_shape_match(prior_preds, eval_gr, tw_tvt, tw_gr,
                          half_win=HALF_WINDOW, search_extra=SEARCH_EXTRA_FT,
                          accept_slack=ACCEPT_SLACK):
    n = len(prior_preds)
    if n == 0:
        return prior_preds

    gr_filled = pd.Series(eval_gr).interpolate(limit_direction="both").to_numpy()
    if np.isnan(gr_filled).all():
        return prior_preds

    refined = prior_preds.copy()
    match_err = np.full(n, np.nan)

    for i in range(n):
        lo_row, hi_row = max(0, i - half_win), min(n, i + half_win + 1)
        local_gr = gr_filled[lo_row:hi_row]
        L = len(local_gr)
        if np.isnan(local_gr).any() or L < 3:
            continue

        center_prior = prior_preds[i]
        lo_idx = np.searchsorted(tw_tvt, center_prior - search_extra)
        hi_idx = np.searchsorted(tw_tvt, center_prior + search_extra)
        if hi_idx - lo_idx < L:
            continue

        seg_gr = tw_gr[lo_idx:hi_idx]
        seg_tvt = tw_tvt[lo_idx:hi_idx]
        windows = np.lib.stride_tricks.sliding_window_view(seg_gr, L)
        sse = np.sum((windows - local_gr[None, :]) ** 2, axis=1)
        best = int(np.argmin(sse))

        center_offset = i - lo_row
        refined[i] = seg_tvt[best + center_offset]
        match_err[i] = sse[best] / L

    valid = ~np.isnan(match_err)
    if valid.sum() == 0:
        return prior_preds

    thresh = np.nanmedian(match_err[valid]) * accept_slack
    keep = valid & (match_err <= thresh)
    return np.where(keep, refined, prior_preds)


FEATURE_COLS = [
    "MD", "X", "Y", "Z", "GR", "linear_prior", "windowed_match",
    "match_minus_prior", "dist_from_known_boundary", "eval_zone_frac",
    "known_zone_rows",
]


def build_well_features(well, split_dir, has_target):
    """One row per eval-zone position for this well. has_target=True for train
    wells (adds the true TVT as 'target' for training); False for test wells."""
    hz = pd.read_csv(os.path.join(split_dir, f"{well}__horizontal_well.csv")).reset_index(drop=True)
    tw_path = os.path.join(split_dir, f"{well}__typewell.csv")
    tw = pd.read_csv(tw_path).dropna(subset=["TVT", "GR"]).sort_values("TVT")

    known = hz[hz["TVT_input"].notna()]
    eval_rows = hz[hz["TVT_input"].isna()]
    if len(eval_rows) == 0:
        return None

    linear_prior = linear_prior_predict(known, eval_rows)

    if len(tw) >= HALF_WINDOW * 2 + 1:
        tw_tvt = tw["TVT"].to_numpy()
        tw_gr = tw["GR"].to_numpy()
        eval_gr = eval_rows["GR"].to_numpy()
        windowed_match = windowed_shape_match(linear_prior, eval_gr, tw_tvt, tw_gr)
    else:
        windowed_match = linear_prior.copy()

    known_md_max = known["MD"].max() if len(known) else eval_rows["MD"].min()
    n_eval = len(eval_rows)

    df = pd.DataFrame({
        "well": well,
        "row_idx": eval_rows.index.to_numpy(),
        "MD": eval_rows["MD"].to_numpy(),
        "X": eval_rows["X"].to_numpy(),
        "Y": eval_rows["Y"].to_numpy(),
        "Z": eval_rows["Z"].to_numpy(),
        "GR": eval_rows["GR"].to_numpy(),
        "linear_prior": linear_prior,
        "windowed_match": windowed_match,
        "match_minus_prior": windowed_match - linear_prior,
        "dist_from_known_boundary": eval_rows["MD"].to_numpy() - known_md_max,
        "eval_zone_frac": (np.arange(n_eval) + 1) / n_eval,
        "known_zone_rows": len(known),
    })

    if has_target and "TVT" in hz.columns:
        df["target"] = eval_rows["TVT"].to_numpy()

    return df


def build_dataset(split_dir, wells, has_target):
    frames = []
    failed = []
    for well in wells:
        try:
            f = build_well_features(well, split_dir, has_target)
            if f is not None:
                frames.append(f)
        except Exception as exc:  # noqa: BLE001 - one bad well must not kill the run
            print(f"  well {well} failed ({exc}); skipping")
            failed.append(well)
    if not frames:
        return pd.DataFrame(), failed
    return pd.concat(frames, ignore_index=True), failed

In [ ]:
t0 = time.time()
train_wells = list_wells(TRAIN_DIR)
print(f"Building TRAIN features for {len(train_wells)} wells...")
train_data, train_failed = build_dataset(TRAIN_DIR, train_wells, has_target=True)
print(f"Train dataset: {train_data.shape}, {len(train_failed)} wells failed, "
      f"built in {time.time()-t0:.1f}s")

train_data = train_data.dropna(subset=["target"])
print(f"After dropping rows with no target: {train_data.shape}")

In [ ]:
# Train the FINAL model on ALL training data - the 5-fold GroupKFold validation
# (RMSE 52.90) already happened locally; this is the deployment fit.
X_train = train_data[FEATURE_COLS]
y_train = train_data["target"].to_numpy()

model = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
model.fit(X_train, y_train)
print("Model trained on", len(train_data), "rows.")

# Global fallback for any test well we fail to process at all.
GLOBAL_FALLBACK = float(train_data["target"].mean())
print(f"GLOBAL_FALLBACK = {GLOBAL_FALLBACK:.2f}")

In [ ]:
test_wells = list_wells(TEST_DIR)
print(f"Building TEST features for {len(test_wells)} wells...")
test_data, test_failed = build_dataset(TEST_DIR, test_wells, has_target=False)
print(f"Test dataset: {test_data.shape}, {len(test_failed)} wells failed")

rows = []
if len(test_data) > 0:
    X_test = test_data[FEATURE_COLS]
    # HistGradientBoostingRegressor handles NaN features natively; no explicit
    # imputation needed, but guard predict() itself so one bad batch can't kill the run.
    try:
        preds = model.predict(X_test)
    except Exception as exc:  # noqa: BLE001
        print(f"batch predict failed ({exc}); falling back to GLOBAL_FALLBACK for all test rows")
        preds = np.full(len(test_data), GLOBAL_FALLBACK)

    for well, row_idx, pred in zip(test_data["well"], test_data["row_idx"], preds):
        rows.append({"id": f"{well}_{row_idx}", "tvt": pred})

# Wells that failed feature-building entirely still need SOME prediction for
# every id sample_submission expects - handled by the reconciliation step below.
submission = pd.DataFrame(rows, columns=["id", "tvt"])
print(f"built {len(submission)} predictions across "
      f"{len(test_wells) - len(test_failed)} wells ({len(test_failed)} wells failed)")
submission.head()

In [ ]:
# Reconcile against the sample submission without asserting/crashing - any gap
# gets filled with GLOBAL_FALLBACK and printed as a warning instead of killing
# the run. A degraded-but-complete submission.csv beats a perfect run that
# never finishes and writes nothing (this bit us on the Stage 2 notebook).
sample = pd.read_csv(SAMPLE_SUBMISSION)
submission = submission.drop_duplicates(subset="id", keep="first")

merged = sample[["id"]].merge(submission, on="id", how="left")
n_missing = merged["tvt"].isna().sum()
if n_missing > 0:
    print(f"WARNING: {n_missing} required ids had no prediction - filling with "
          f"GLOBAL_FALLBACK ({GLOBAL_FALLBACK:.2f})")
    merged["tvt"] = merged["tvt"].fillna(GLOBAL_FALLBACK)

extra_ids = set(submission["id"]) - set(sample["id"])
if extra_ids:
    print(f"NOTE: {len(extra_ids)} predicted ids aren't in sample_submission - dropped "
          f"(harmless, e.g. from a visible-example test well not in the real hidden set)")

assert len(merged) == len(sample), f"row count still off: {len(merged)} vs {len(sample)}"
assert merged["tvt"].notna().all(), "still have NaNs after fallback fill - should be impossible"

submission = merged
print("Reconciliation complete - submission.csv will have every required id.")
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv:", submission.shape)
print(f"Total runtime: {time.time()-t0:.1f}s")